In [3]:
!pip install minio pandas tqdm

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from minio import Minio
import pandas as pd
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import threading

# Configurações do MinIO
MINIO_ENDPOINT = "minio-pc88ws4wgwswgso8ggcckcsc.sendvers.pro"
MINIO_ACCESS_KEY = "tZH9U78VVEfrOen77sz4"
MINIO_SECRET_KEY = "QubbFU5AFjYfiQO2og5CULG4rDZLvhBvzAlZNIwI"
MINIO_BUCKET = "processos-sei"
MINIO_PREFIX = ""  # ajuste conforme necessário

# Inicializa cliente
client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=True
)

# Output CSVs
REMETIDO_RECEBIDO_CSV = "remetido_recebido_gasec.csv"
PROCESSADOS_TXT = "processados.txt"
CSV_LOCK = threading.Lock()
TXT_LOCK = threading.Lock()

# Função para registrar arquivo processado
def registrar_arquivo_processado(nome_arquivo, arquivo_log=PROCESSADOS_TXT):
    with TXT_LOCK:
        with open(arquivo_log, 'a', encoding='utf-8') as f:
            f.write(nome_arquivo + '\n')

# Carregar arquivos já processados
if os.path.exists(PROCESSADOS_TXT):
    with open(PROCESSADOS_TXT, 'r', encoding='utf-8') as f:
        arquivos_processados = set(l.strip() for l in f.readlines())
else:
    arquivos_processados = set()    

# Função para processar e salvar remetido/recebido para unidade específica
# Função para processar e salvar processos tramitados para unidade específica (apenas número do processo)
unidade_destino = 'SEFAZ-PI/GASEC/CGFR'
def process_file(obj):
    object_name = obj.object_name
    protocol_name = object_name.split('/')[1].split('.j')[0]

    if object_name in arquivos_processados:
        #print(f"Ignorando já processado: {object_name}")
        return
    local_file = f"temp_{threading.get_ident()}_{os.path.basename(object_name)}"
    try:
        # Baixar arquivo
        client.fget_object(MINIO_BUCKET, object_name, local_file)
        # Ler e processar JSON
        with open(local_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        # Montar DataFrame
        #print(f'ordenando historico {protocol_name}')
        try:
            rows = []
            for andamento in data[::-1]:
                rows.append({
                    'Processo': protocol_name,
                    'Tarefa': andamento.get('Tarefa'),
                    'Unidade': andamento.get('Unidade').get('Sigla'),
                    'Descricao': andamento.get('Descricao'),
                    'DataHora': andamento.get('DataHora')
                })
            if rows == []:
                print(f"Nenhum andamento encontrado em {object_name}.")
                #registrar_arquivo_processado(object_name)
                return
            print(f'Protocolo {object_name}, has {len(rows)} andamentos')
            #print('criando dataframe')
            df = pd.DataFrame(rows)
            if df.empty:
                print(f'Erro no parsing do dataframe de andamento encontrado em {object_name}.')
                raise Exception
        except Exception as e:
            print(f'Erro no ordenamento do historico de {object_name}:', e)
            return
        
        # 1. Filter all df rows that column Tarefa is not 'PROCESSO-REMETIDO-UNIDADE' or 'PROCESSO-RECEBIDO-UNIDADE'
        # 2. Keep only rows that contains (column Tarefa == 'PROCESSO-REMETIDO-UNIDADE' and column Unidade == unidade_destino)
        # OR (column Tarefa == 'PROCESSO-RECEBIDO-UNIDADE' and column Unidade == unidade_destino) 
        # OR (column Tarefa == 'PROCESSO-REMETIDO-UNIDADE' and column Unidade contains "SEAD-PI/" and column Descricao contains "Processo remetido pela unidade" and contains "unidade_destino")
        
        # if (column Tarefa == 'PROCESSO-REMETIDO-UNIDADE' and column Unidade == unidade_destino) its true than set insights['processo'] as protocol_name and insights['tramitado-sead-cgfr'] from column "DataHora"
        # if the above its true for and (column Tarefa == 'PROCESSO-RECEBIDO-UNIDADE' and column Unidade == unidade_destino) its true than setinsights['recebido-cgfr'] == 1 else 0 and insights['data-recebido-cgfr'] from column "DataHora"
        # if the 2 conditions above are true and (column Tarefa == 'PROCESSO-REMETIDO-UNIDADE' and column Unidade contains "SEAD-PI/" and column Descricao contains "Processo remetido pela unidade" and contains "unidade_destino") its true than set insights['devolvido-cgfr-sead'] == 1 else 0 and insights['data_devolvido-cgfr-sead'] from column "DataHora"
        # if insights != {} save into a new row in a text file REMETIDO_RECEBIDO_CSV

        try:
            insights = {}
            # print(f'Filtrando tarefas para {protocol_name}')

            # Condition 1 (Trigger): Find the first instance of the process being remitted to the destination unit.
            remetido_para_destino_event = df[
                (df['Tarefa'] == 'PROCESSO-REMETIDO-UNIDADE') &
                (df['Unidade'] == unidade_destino)
            ].head(1)

            # The entire analysis is contingent on this first condition being met.
            if not remetido_para_destino_event.empty:
                insights['processo'] = protocol_name
                insights['tramitado-sead-cgfr'] = remetido_para_destino_event['DataHora'].iloc[0]

                # Condition 2: Check if the process was ever marked as received by the destination unit.
                recebido_na_unidade_event = df[
                    (df['Tarefa'] == 'PROCESSO-RECEBIDO-UNIDADE') &
                    (df['Unidade'] == unidade_destino)
                ].head(1)

                if not recebido_na_unidade_event.empty:
                    insights['recebido-cgfr'] = 1
                    insights['data-recebido-cgfr'] = recebido_na_unidade_event['DataHora'].iloc[0]
                else:
                    insights['recebido-cgfr'] = 0
                    insights['data-recebido-cgfr'] = None

                # Condition 3: Check if the process was remitted from SEAD to the destination unit.
                devolvido_pela_sead_event = df[
                    (df['Tarefa'] == 'PROCESSO-REMETIDO-UNIDADE') &
                    (df['Unidade'].str.contains("SEAD-PI/", na=False)) &
                    (df['Descricao'].str.contains("Processo remetido pela unidade", na=False)) &
                    (df['Descricao'].str.contains(unidade_destino, na=False))
                ].head(1)

                if not devolvido_pela_sead_event.empty:
                    insights['devolvido-cgfr-sead'] = 1
                    insights['data_devolvido-cgfr-sead'] = devolvido_pela_sead_event['DataHora'].iloc[0]
                else:
                    insights['devolvido-cgfr-sead'] = 0
                    insights['data_devolvido-cgfr-sead'] = None

            # If insights were generated, save them to the CSV file.
            if insights:
                # Create a DataFrame from the insights dictionary.
                df_result = pd.DataFrame([insights])
                
                # Define column order for consistent CSV output.
                column_order = [
                    'processo', 'tramitado-sead-cgfr', 'recebido-cgfr', 
                    'data-recebido-cgfr', 'devolvido-cgfr-sead', 'data_devolvido-cgfr-sead'
                ]
                df_result = df_result[column_order]

                # Use a lock for thread-safe CSV writing.
                with CSV_LOCK:
                    file_exists = os.path.exists(REMETIDO_RECEBIDO_CSV)
                    df_result.to_csv(
                        REMETIDO_RECEBIDO_CSV,
                        sep=';',
                        index=False,
                        mode='a',
                        header=not file_exists
                    )
                print(f'Protocolo {protocol_name} adicionado ao arquivo: {REMETIDO_RECEBIDO_CSV}')
            #else:
                # print(f"Nenhum trâmite qualificado para a unidade {unidade_destino} encontrado em {object_name}.")

            # Registrar arquivo processado
            registrar_arquivo_processado(object_name)

        except Exception as e:
            print(f"Erro processando {object_name}: {e}, !! andamento: {andamento if 'andamento' in locals() else 'N/A'}, !! {rows if 'rows' in locals() else 'N/A'}")
    finally:
        if os.path.exists(local_file):
            os.remove(local_file)
            pass


In [10]:
# Listar todos os objetos
objects = list(client.list_objects(MINIO_BUCKET, prefix=MINIO_PREFIX, recursive=True))
print(f"Total de arquivos encontrados: {len(objects)}")

# Processamento paralelo
NUM_WORKERS = 20  # ajuste conforme CPU / rede
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = [executor.submit(process_file, obj) for obj in objects]
    for _ in tqdm(as_completed(futures), total=len(futures)):
        pass

print("Processamento completo. CSV gerado em:", REMETIDO_RECEBIDO_CSV)

Total de arquivos encontrados: 5179


  0%|          | 0/5179 [00:00<?, ?it/s]

Protocolo 2024/00002.003487_2024-48.json, has 30 andamentos
Protocolo 00002.003487_2024-48 adicionado ao arquivo: remetido_recebido_gasec.csv
Processamento completo. CSV gerado em: remetido_recebido_gasec.csv
